# Canonical probabilities for finite PIAB1D levels

**Prerequisite:** [Canonical probabilities on finite PIAB1D levels](../../../docs/lecture-notes/qm/piab1d/piab1d__canonical_probabilities.md)

This laboratory preserves the single-particle canonical-occupation graph from the computational PIAB prototype while separating it from many-fermion Fermi--Dirac occupations.

## Learning objectives

1. Evaluate a finite analytical PIAB1D energy inventory with physical units.
2. Construct normalized canonical probabilities for one modeled particle.
3. Check normalization and the retained high-energy tail.
4. Relate the energy scale to the resulting probability distribution.

The electron mass, 10 nm box, and 300 K temperature form an illustrative model. They are not evidence for a material or device.


## Finite-level canonical model

For represented energies $E_n$ and thermal energy $k_B T$,

$$
P_n=\frac{\exp[-(E_n-E_{\min})/(k_B T)]}
{\sum_m\exp[-(E_m-E_{\min})/(k_B T)]}.
$$

Subtracting the minimum energy is a numerically stable common shift and does not change the normalized probabilities. These $P_n$ sum to one because they describe mutually exclusive levels of one modeled particle. They are not Fermi--Dirac expected occupations.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.constants import SI
from projectkoios.physkit.qm.piab1d.base import Piab1D
from projectkoios.physkit.qm.piab1d.tise_analytical.solution import (
    Piab1DAnalyticalSolution,
)
from projectkoios.physkit.thermal.statmech.qm import (
    CanonicalLevelProbabilityModel,
)
from projectkoios.physkit.units import (
    MODEL_SYSTEM_UNIT_CONVERTER,
    PhysicalUnit,
    ScalarQuantity,
    UnitSystem,
)

In [ ]:
temperature_kelvin = 300.0
level_count = 40
model = Piab1D(
    length=10.0e-9,
    mass=SI.me0,
    unit_system=UnitSystem.SI,
)
stationary_states = Piab1DAnalyticalSolution(model=model).evaluate(count=level_count)
thermal_energy = ScalarQuantity(
    magnitude=SI.k_B * temperature_kelvin,
    unit=model.unit_system.energy_unit,
)
canonical_evaluation = CanonicalLevelProbabilityModel(
    thermal_energy=thermal_energy,
).evaluate(
    energies=stationary_states.energies,
)
probabilities = canonical_evaluation.probabilities.magnitude
assert np.isclose(np.sum(probabilities), 1.0, rtol=0.0, atol=1.0e-15)
assert np.all(probabilities[1:] < probabilities[:-1])

In [ ]:
energies_electron_volts = MODEL_SYSTEM_UNIT_CONVERTER.convert_vector(
    stationary_states.energies,
    PhysicalUnit(expression="electron_volt"),
)
thermal_energy_electron_volts = MODEL_SYSTEM_UNIT_CONVERTER.convert_scalar(
    thermal_energy,
    PhysicalUnit(expression="electron_volt"),
)
shown_level_count = 15
retained_tail_probability = float(np.sum(probabilities[shown_level_count:]))
assert retained_tail_probability < 1.0e-12
thermal_energy_electron_volts.magnitude, retained_tail_probability

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.75))
shown_quantum_numbers = stationary_states.quantum_numbers[:shown_level_count]

axes[0].plot(
    shown_quantum_numbers,
    (
        energies_electron_volts.magnitude[:shown_level_count]
        / thermal_energy_electron_volts.magnitude
    ),
    marker="o",
)
axes[0].axhline(
    1.0,
    color="black",
    linestyle="--",
    label=r"thermal scale $k_B T$",
)
axes[0].set(
    xlabel="quantum number $n$",
    ylabel=r"$E_n/(k_B T)$",
    title="Energy scale relative to temperature",
    xticks=shown_quantum_numbers,
)
axes[0].grid(True)
axes[0].legend()

axes[1].bar(
    shown_quantum_numbers,
    probabilities[:shown_level_count],
)
axes[1].set(
    xlabel="quantum number $n$",
    ylabel="canonical probability",
    title="Single-particle canonical distribution",
    xticks=shown_quantum_numbers,
)
axes[1].grid(True, axis="y")

figure.tight_layout()
plt.show()

## Interpretation

The energy spacing grows as $n^2$, so only the lowest levels have appreciable canonical probability at the selected thermal scale. The displayed bars preserve the original finite-level thermal presentation while the adjacent spectrum makes its scale explicit. The omitted probability above the first 15 levels is checked rather than assumed.

This distribution is normalized over one particle's mutually exclusive levels. The expected occupations of a finite fermion population require a chemical potential and are treated separately in `piab1d__fermions.ipynb`.

## Exercises

1. Increase the box length and explain why more levels acquire appreciable probability.
2. Increase the temperature and monitor the retained tail probability.
3. Determine how many levels are needed to retain a declared probability fraction.
4. Compare the canonical probabilities with Fermi--Dirac occupations without equating their normalizations.
